# EFFICIENT CODING

In [ ]:
import math
import sys
from dataclasses import replace, asdict
from typing import Tuple

import pandas as pd
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

from pathtracer import Renderer, lattice_preview, display_encode

sys.path.insert(0, "..")
from spectral import lattice_geometry, row_twiddle, fft, ifft, bin_freq, alias_radius_sq

In [ ]:
def device_mapper() -> torch.device:
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

def host(x: torch.Tensor) -> np.ndarray: return x.cpu().numpy()

DEVICE = device_mapper()

---
# DATA
Renders the test scene once: a colour image for PCA and a brightness image for every other section.

- `beauty` and `luminance` are scene-referred linear values in cd/m^2, not display codes
- `rendercam` is the single source of camera state: later sections read their lattice geometry from it
- `show_sky` makes primary misses return environment radiance, so every pixel sees light rather than a black void
- Change the camera, sky or AOVs here and rerun all; only STATISTICS renders again, its second half under the same settings

In [ ]:
scene = Renderer("scenes/cornell.json")
rendercam = replace(scene.default_camera)
frame = scene.render(camera=rendercam, aovs=("beauty", "luminance"), show_sky=True)

In [ ]:
stats = {k: dict(shape=v.shape, dtype=v.dtype, min=v.min(), max=v.max(), mean=v.mean()) for k, v in frame.items()}
display(pd.DataFrame(stats).T, pd.Series(asdict(rendercam), name="camera").to_frame())

---
# IMAGE
The render as a screen would show it, with its 8-bit pixel counts and its red, green and blue channels.

- `lattice_preview` draws each pixel as its true cell (hexagon or square), so the hex row offset is visible
- Stats and channels use `display_encode` output: camera exposure, OCIO display transform, dither, then 8-bit quantise
- Statistics therefore describe the displayed image, not scene radiance; dither fills codes a plain quantise would leave empty

In [ ]:
rgb = torch.from_numpy(frame["beauty"]).to(DEVICE)

plt.imshow(display_encode(lattice_preview(host(rgb), rendercam, scale=8), camera=rendercam))
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
def pixel_distribution(img: torch.Tensor) -> Tuple[torch.Tensor, tuple]:
    """Counts per value 0-255 (index, all, then each channel) and summary stats of a (height, width, 1 or 3) uint8 image."""
    height, width, channels = img.shape
    values = img.long()
    pixel_count = torch.stack([
        torch.arange(256, device=img.device),
        torch.bincount(values.flatten(), minlength=256),
        *(torch.bincount(values[..., c].flatten(), minlength=256) for c in range(channels)),
    ])

    counts = pixel_count[1]
    occupied = counts.nonzero().flatten()
    pixels = height * width
    median = (counts.cumsum(0) > counts.sum() / 2).int().argmax().item()
    nonzero = [occupied[0].item(), occupied[-1].item()]
    empty_bins = (counts == 0).sum().item()
    unique_colors = torch.unique(img.reshape(pixels, channels), dim=0).shape[0]
    least_frequent = occupied[counts[occupied].argmin()].item()
    level = img.float()

    return pixel_count, (pixels, img.min().item(), img.max().item(), least_frequent, counts.argmax().item(), round(level.mean().item(), 2), round(level.std(correction=0).item(), 2), median, nonzero, empty_bins, unique_colors)

In [ ]:
encoded = torch.from_numpy(display_encode(frame["beauty"], camera=rendercam)).to(DEVICE)

_, stat = pixel_distribution(encoded)

labels = ("shape", "pixels", "min", "max", "least_frequent", "most_frequent", "mean", "std", "median", "nonzero", "empty_bins", "unique_colours")
display(pd.Series((tuple(encoded.shape), *stat), index=labels, name="beauty").to_frame())

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=[18, 5])
for c, name in enumerate(("R", "G", "B")):
    ax[c].imshow(host(encoded[..., c]), cmap="gray")
    ax[c].set_title(name)
    ax[c].set_xticks([]), ax[c].set_yticks([])
plt.tight_layout()

---
# SPECTRAL
Splits the brightness image into waves of different sizes and directions: how much each contributes, and how much is render noise.

- Lattice-aware DFT: a per-row twiddle moves odd rows to their true positions, so square and hex share one path
- Frequencies are cycles per column pitch; each bin is folded to its nearest reciprocal-lattice point for its true radius and angle
- Radial power excludes DC and stops at the lattice Nyquist; the spatial frequency curve is f·P(f), so its peak is the dominant scale
- Phase is referenced to the luminance centroid, so it does not depend on where the scene sits in frame
- Log magnitude is dB re the peak bin; zero bins are NaN and left out of the noise floor histogram

In [ ]:
row_pitch, row_shift = lattice_geometry(rendercam)
img = torch.from_numpy(frame["luminance"][..., 0]).to(DEVICE)
twiddle = row_twiddle(img.shape, row_shift, DEVICE)
spectrum = fft(img, twiddle)
fy, fx = bin_freq(img.shape, row_pitch, DEVICE)
radius_sq, true_fy, true_fx = alias_radius_sq(fy, fx, row_pitch, row_shift)
radius = radius_sq.sqrt()
mag = spectrum.abs()

bin_width = 1.0 / min(img.shape[1], img.shape[0] * row_pitch)
nyquist = 0.5 * min(1.0 / row_pitch, math.hypot(1.0, row_shift / row_pitch))
ring_index = (radius / bin_width).long()
power = torch.zeros(int(ring_index.max()) + 1, device=DEVICE).index_add_(0, ring_index.flatten(), torch.where(radius > 0, mag ** 2, 0).flatten())
edges = torch.arange(power.numel() + 1, device=DEVICE) * bin_width
centres = 0.5 * (edges[1:] + edges[:-1])
inside = edges[1:] <= nyquist
log_power = centres * power
peak = torch.where(inside, log_power, 0).argmax()
print(f"Frequency at which power per log frequency is greatest: {centres[peak].item():.4f} cycles/pitch")

total = img.sum()
cx = (torch.arange(img.shape[1], device=DEVICE, dtype=img.dtype) @ img.sum(0) + row_shift * img[1::2].sum()) / total
cy = row_pitch * (torch.arange(img.shape[0], device=DEVICE, dtype=img.dtype) @ img.sum(1)) / total
phase = torch.angle(spectrum * torch.exp(2j * math.pi * (fx[None, :] * cx + fy[:, None] * cy)))
log_mag = torch.where(mag > 0, 20 * torch.log10(mag / mag.max()), torch.nan)

In [ ]:
fig, (ax_lum, ax_mag, ax_phase) = plt.subplots(1, 3, figsize=[30,10])
ax_lum.imshow(host(img))
ax_lum.set_title("Luminance")

ax_mag.imshow(host(torch.fft.fftshift(log_mag)), cmap="binary")
ax_mag.set_title("Log magnitude")

ax_phase.imshow(host(torch.fft.fftshift(phase)), cmap="binary")
ax_phase.set_title("Phase")

In [ ]:
power_2d = mag ** 2
theta = torch.atan2(true_fy, true_fx)

fig, ax = plt.subplots(1, 5, figsize=[25, 5])

ax[0].plot(host(centres[inside]), host(log_power[inside]), lw=1)
ax[0].axvline(centres[peak].item(), color="r", lw=1)
ax[0].set_title("Spatial frequency")
ax[0].set_xlabel("Radius (cycles/pitch)"), ax[0].set_ylabel("Power per log frequency")
ax[0].set_ylim(bottom=0)

ax[1].plot(host(edges[1:]), host(power.cumsum(0) / power.sum()), lw=1)
ax[1].set_title("Cumulative energy")
ax[1].set_xlabel("Radius (cycles/pitch)"), ax[1].set_ylabel("Fraction of AC power")

ax[2].semilogy(host(torch.fft.fftshift(fx)), host(torch.fft.fftshift(power_2d.sum(0))), lw=1, label="fx")
ax[2].semilogy(host(torch.fft.fftshift(fy)), host(torch.fft.fftshift(power_2d.sum(1))), lw=1, label="fy")
ax[2].set_title("Marginal spectra")
ax[2].set_xlabel("Frequency (cycles/pitch)"), ax[2].set_ylabel("Power")
ax[2].legend()

ax[3].hist(host(log_mag[log_mag.isfinite()]), bins=100)
ax[3].set_title("Noise floor")
ax[3].set_xlabel("Magnitude (dB re peak)"), ax[3].set_ylabel("Bins")

ring = (radius > 0) & (radius <= nyquist)
ax[4].hist(host(theta[ring]), bins=36, range=(-math.pi, math.pi), weights=host(power_2d[ring]))
ax[4].set_title("Angular power")
ax[4].set_xlabel("Angle (rad)"), ax[4].set_ylabel("Power")

for a in ax: a.grid(alpha=0.25)

---
# STATISTICS
How often each brightness occurs, and how alike nearby pixels are: the predictability (redundancy) an efficient code should remove.

- Input is `log(L + dark)`, the photoreceptor's Weber compression; dark light = RMS render noise of the mean (Barlow), finite at L = 0
- A second render with an independent sampler scramble (`seed=2`) shares the signal only, so WHITENING can split signal from noise
- First order: the shuffled twin has exactly the same histogram but no scene, so pixels are not independent (Attneave 1954)
- Second order: pixels `d` column pitches apart along a row, the same true distance on square and hex lattices
- Separations 1, 2 and 4 replicate Simoncelli & Olshausen 2001 fig. 1: correlation falls with distance but stays high, i.e. redundancy
- Correlation vs distance is the normalised row autocovariance, the Fourier pair of the power spectrum summed over fy (Wiener-Khinchin)

In [ ]:
half = torch.from_numpy(scene.render(camera=rendercam, aovs=("luminance",), seed=2, show_sky=True)["luminance"][..., 0]).to(DEVICE)
dark = ((img - half) ** 2 / 4).mean().sqrt()
halves = torch.log(torch.stack((img, half)) + dark)
log_luminance = halves.mean(0)
generator = torch.Generator(device=DEVICE).manual_seed(0)
shuffled = log_luminance.flatten()[torch.randperm(log_luminance.numel(), generator=generator, device=DEVICE)].reshape(log_luminance.shape)

correlation = lambda x, d: torch.corrcoef(torch.stack((x[:, :-d].flatten(), x[:, d:].flatten())))[0, 1]
distances = range(1, img.shape[1] // 2 + 1)
falloff = torch.stack([correlation(log_luminance, d) for d in distances])

In [ ]:
previews = [lattice_preview(host(x[..., None]), rendercam) for x in (log_luminance, shuffled)]
fig, ax = plt.subplots(1, 3, figsize=[24, 5], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews), 1])
for a, p, name in zip(ax, previews, ("Log luminance", "Shuffled")):
    a.imshow(p, cmap="gray")
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

ax[2].hist(host(log_luminance.flatten()), bins="auto", histtype="step")
ax[2].set_title("Histogram (shared)")
ax[2].set_xlabel("log(L + dark)"), ax[2].set_ylabel("Pixels")
ax[2].set_box_aspect(1), ax[2].grid(alpha=0.25)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=[24, 6])
for a, d in zip(ax, (1, 2, 4)):
    a.plot(host(log_luminance[:, :-d].flatten()), host(log_luminance[:, d:].flatten()), ",")
    a.set_title(f"Separation {d}, r = {falloff[d - 1].item():.3f}")
    a.set_xlabel("x"), a.set_ylabel(f"x + {d}")

ax[3].plot(distances, host(falloff), lw=1)
ax[3].set_title("Correlation vs distance")
ax[3].set_xlabel("Distance (column pitches)"), ax[3].set_ylabel("Correlation")
ax[3].set_ylim(top=1)

for a in ax: a.set_box_aspect(1), a.grid(alpha=0.25)

---
# WHITENING
A model of the eye's first filter: it removes what neighbouring pixels predict about each other, suppresses noise and leaves edges.

- Filter `K = sqrt(R) / (R + N)` (Dayan & Abbott eq. 4.49): Wiener gain `R / (R + N)` times whitening `R^-1/2`
- `R` and `N` are measured, not modelled, from STATISTICS' two independently sampled log luminance renders
- Isotropic like a ganglion cell's centre-surround, so per ring of radius: cross power of the two halves estimates signal `R`, a quarter of their difference power the mean's noise `N`
- Signal falls as a power law `f^-a`, i.e. scale invariance; natural images give a ≈ 2 (Field 1987; Ruderman 1997); fitted where R > N
- Whitened power is `R / (R + N)`: flat (decorrelated) where SNR is high, falling to zero where render noise dominates
- Receptive field is K's impulse response: the localised (ZCA) whitening basis `U Λ^-1/2 Uᵀ`, centre-surround like LGN on/off cells
- PCA's `Λ^-1/2 Uᵀ` whitens equally but gives global Fourier checkerboards; the noise term sets the surround's extent (Atick & Redlich 1992)
- Residual neighbour correlation is the noise roll-off plus the scene's horizontal/vertical bias, which an isotropic filter cannot flatten
- DC is removed, so the output is zero-mean contrast; what survives is edges, the phase structure second-order statistics cannot explain

In [ ]:
first, other = fft(halves, twiddle)

ac = radius > 0
count = torch.zeros(centres.numel(), device=DEVICE).index_add_(0, ring_index[ac], torch.ones_like(radius[ac]))
ring_mean = lambda v: torch.zeros_like(count).index_add_(0, ring_index[ac], v[ac]) / count

signal = ring_mean((first * other.conj()).real).clamp(min=0)
noise = ring_mean((first - other).abs() ** 2 / 4)
gain = signal.sqrt() / (signal + noise)
log_spectrum = (first + other) / 2
transfer = torch.where(ac, gain[ring_index], 0)
whitened_spectrum = log_spectrum * transfer
whitened = ifft(whitened_spectrum, twiddle).real
receptive_field = ifft(transfer, twiddle).real
wavelength = (1 / centres[inside][gain[inside].argmax()]).item()
rows, cols = 2 * math.ceil(wavelength / (2 * row_pitch)), math.ceil(wavelength)
field = torch.roll(receptive_field, (rows, cols), dims=(0, 1))[:2 * rows + 1, :2 * cols + 1]
whitened_power = ring_mean(whitened_spectrum.abs() ** 2)

fit = inside & (signal > noise)
design = torch.stack((centres[fit].log(), torch.ones_like(centres[fit])), 1)
exponent, scale = torch.linalg.lstsq(design, signal[fit].log()[:, None]).solution.flatten().tolist()

print(f"Horizontal neighbour correlation: input {correlation(log_luminance, 1).item():.3f}, whitened {correlation(whitened, 1).item():.3f}")

In [ ]:
previews = [lattice_preview(host(x[..., None]), rendercam) for x in (whitened, field)]
previews[1] = previews[1][:, :round(previews[1].shape[1] * (1 - row_shift / field.shape[1]))]
side = min(previews[1].shape[:2])
top, left = ((n - side) // 2 for n in previews[1].shape[:2])
previews[1] = previews[1][top:top + side, left:left + side]
fig, ax = plt.subplots(1, 4, figsize=[30, 5], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews), 1, 1])

for a, p, name in zip(ax, previews, ("Whitened", "Receptive field")):
    limit = np.abs(p).max()
    a.imshow(p, cmap="gray", vmin=-limit, vmax=limit)
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

ax[2].loglog(host(centres[inside]), host(signal[inside]), lw=1, label="Signal R")
ax[2].loglog(host(centres[inside]), host(noise[inside]), lw=1, label="Noise N")
ax[2].loglog(host(centres[fit]), math.exp(scale) * host(centres[fit]) ** exponent, "k--", lw=1, label=f"Power law f^{exponent:.2f}")
ax[2].set_title("Signal and noise")
ax[2].set_xlabel("Radius (cycles/pitch)"), ax[2].set_ylabel("Power")
ax[2].tick_params(axis="x", which="minor", labelbottom=False)
ax[2].legend()

ax[3].plot(host(centres[inside]), host(whitened_power[inside]), lw=1)
ax[3].set_title("Whitened power")
ax[3].set_xlabel("Radius (cycles/pitch)"), ax[3].set_ylabel("R / (R + N)")
ax[3].set_ylim(bottom=0)

for a in ax[2:]: a.set_box_aspect(1), a.grid(alpha=0.25)

---
# ORIENTED FILTERS
Looks for edges in the whitened image at every orientation: a Gabor bank, a Sobel gradient and a steerable G2/H2 pair.

- Each filter is applied as its analytic spectrum at every bin's true frequency, so one path is exact on square and hex lattices
- Gabor bank: carrier at WHITENING's peak wavelength, 1 octave, 2:1 aspect, 36 orientations over 180° since a cosine Gabor repeats every 180°
- Quadrature energy (cosine and sine partners) is phase-invariant, so edges and lines peak on the feature rather than ripple across it
- Sobel is the classic baseline: central difference along, [1, 2, 1] / 4 across, peak at λ = 4 px
- G2 and its Riesz quadrature H2 share one radial profile, so their hypot is exact quadrature energy, steerable to any angle

In [ ]:
def gabor_filter(sigma_x: float, sigma_y: float, deg: float, k: float, device: torch.device) -> torch.Tensor:
    radius = math.ceil(max(sigma_x, sigma_y) * math.sqrt(-2 * math.log(torch.finfo(torch.float32).eps)))
    r = torch.arange(-radius, radius + 1, dtype=torch.float32, device=device)
    Y, X = torch.meshgrid(r, r, indexing="ij")
    rad = math.radians(deg)
    Xr = math.cos(rad) * X + math.sin(rad) * Y
    Yr = math.cos(rad) * Y - math.sin(rad) * X
    envelope = torch.exp(-Xr ** 2 / (2 * sigma_x ** 2) - Yr ** 2 / (2 * sigma_y ** 2))
    gabor = envelope * torch.cos(k * Xr)
    gabor -= envelope * gabor.sum() / envelope.sum()
    return gabor

In [ ]:
n = 36
row = 6
col = math.ceil(n / row)

k = 2 * math.pi / wavelength
octaves = 1
sigma_x = wavelength / math.pi * math.sqrt(math.log(2) / 2) * (2 ** octaves + 1) / (2 ** octaves - 1)
sigma_y = 2 * sigma_x

plt.figure(figsize=(18, 18))
for i in range(n):
    deg = 180 * i / n
    plt.subplot(row, col, i + 1)
    plt.imshow(host(gabor_filter(sigma_x, sigma_y, deg, k, DEVICE)), cmap="gray")
    plt.title(f"Angle: {deg:g}")
    plt.axis("off")

plt.tight_layout()

In [ ]:
# Analytic Gabor spectrum at each bin's true frequency, so the bank is exact on square and hex lattices
theta = torch.pi * torch.arange(n, device=DEVICE)[:, None, None] / n
along = 2 * torch.pi * (torch.cos(theta) * true_fx + torch.sin(theta) * true_fy)
across = 2 * torch.pi * (torch.cos(theta) * true_fy - torch.sin(theta) * true_fx)
bump = lambda centre: torch.exp(-((sigma_x * (along - centre)) ** 2 + (sigma_y * across) ** 2) / 2)
bank = 2 * (bump(k) - math.exp(-(sigma_x * k) ** 2 / 2) * bump(0))
responses = ifft(whitened_spectrum * bank, twiddle)

fig, ax = plt.subplots(row, col, figsize=(18, 10))
for i, (a, response) in enumerate(zip(ax.flat, responses)):
    a.imshow(lattice_preview(host(response.real[..., None]), rendercam)[..., 0], cmap="gray")
    a.set_title(f"Angle: {180 * i / n:g}")
    a.set_xticks([]), a.set_yticks([])

plt.tight_layout()

In [ ]:
# Quadrature energy per orientation; doubled-angle vector sum gives orientation (hue) and oriented strength (value)
energy = responses.abs()
z = (energy * torch.exp(2j * theta)).sum(0)
hsv = torch.stack([(z.angle() / (2 * torch.pi)) % 1, torch.ones_like(energy[0]), z.abs() / z.abs().max()], -1)

plt.figure(figsize=(10, 5))
plt.imshow(lattice_preview(hsv_to_rgb(host(hsv)), rendercam, scale=8))
plt.title("Edges: hue = orientation, value = strength")
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
gabor_edges = responses.abs().amax(0)

plt.figure(figsize=(10, 5))
plt.imshow(lattice_preview(host(gabor_edges[..., None]), rendercam, scale=8)[..., 0], cmap="gray")
plt.title("Gabor filter")
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
# Sobel's frequency response at each bin's true frequency: central difference along, [1, 2, 1] / 4 smoothing across
sobel = lambda u, v: 1j * torch.sin(2 * torch.pi * u) * torch.cos(torch.pi * v) ** 2
gx, gy = (ifft(whitened_spectrum * sobel(u, v), twiddle).real for u, v in ((true_fx, true_fy), (true_fy, true_fx)))
sobel_edges = torch.hypot(gx, gy)

plt.figure(figsize=(10, 5))
plt.imshow(lattice_preview(host(sobel_edges[..., None]), rendercam, scale=8)[..., 0], cmap="gray")
plt.title("Sobel kernel")
plt.xticks([]), plt.yticks([])
plt.tight_layout()

In [ ]:
# G2 (cos²) and its Riesz quadrature H2 (cos³) share one radial profile, so hypot is phase-invariant edge energy at every theta
band = 1 / 4  # cycles per pixel: Sobel's peak, lambda = 4 px
radial = radius ** 2 * torch.exp(-(radius / band) ** 2)
cos = torch.cos(torch.atan2(true_fy, true_fx) - theta)
g2 = ifft(whitened_spectrum * radial * cos ** 2, twiddle).real
h2 = ifft(whitened_spectrum * radial * cos ** 3, twiddle).imag
steerable_edges = torch.hypot(g2, h2).amax(0)

plt.figure(figsize=(10, 5))
plt.imshow(lattice_preview(host(steerable_edges[..., None]), rendercam)[..., 0], cmap="gray")
plt.title("G2/H2")
plt.xticks([]), plt.yticks([])
plt.tight_layout()

---
# PHASE
Keeps how strong each wave is but scrambles where it sits: the scene turns to smoke, so edges live in the waves' positions.

- Oppenheim & Lim 1981: edges need the phases of many frequencies aligned at one place; magnitude alone fixes only pairwise correlation
- Each pair shares one power spectrum exactly, so pairwise pixel correlations match; the scrambled twin keeps none of the edges
- Random phase is the angle of the lattice DFT of white Gaussian noise: uniform, and conjugate-symmetric, so the image stays real
- A random-phase twin is the Gaussian image model's view (Simoncelli & Olshausen 2001): right pairwise correlations, nothing more, so smoke
- DC is removed from every image, so all four are zero-mean and compare like for like

In [ ]:
generator = torch.Generator(device=DEVICE).manual_seed(0)
random_phase = torch.sgn(fft(torch.randn(img.shape, generator=generator, device=DEVICE), twiddle))
log_ac = torch.where(ac, log_spectrum, 0)

images = {
    "Log luminance": ifft(log_ac, twiddle).real,
    "Random phase": ifft(log_ac.abs() * random_phase, twiddle).real,
    "Whitened": whitened,
    "Whitened, random phase": ifft(whitened_spectrum.abs() * random_phase, twiddle).real,
}

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=[20, 10])
for a, (name, image) in zip(ax.ravel(), images.items()):
    limit = image.abs().max().item()
    a.imshow(lattice_preview(host(image[..., None]), rendercam), cmap="gray", vmin=-limit, vmax=limit)
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

---
# SPARSITY
Edges make images peaky, mostly near zero with rare large values; scrambled images lose this and become bell-shaped noise.

- Excess kurtosis `m4 / m2^2 - 3` is 0 for a Gaussian and 3 for a Laplace; positive means super-Gaussian, i.e. sparse (Field 1994)
- A random-phase twin sums many waves of independent phase, so the central limit theorem pulls it towards Gaussian
- That pull is weak when power sits in a few low frequencies, so the log luminance twin can stay measurably off zero
- Whitening spreads power over all frequencies: its twin is Gaussian, while the true-phase image keeps the heavy tails of its edges
- Histograms are in standard deviations (every image is zero-mean); the references are the unit-variance Gaussian and Laplace densities

In [ ]:
kurtosis = lambda x, dim=None: (x ** 4).mean(dim) / x.var(dim, correction=0) ** 2 - 3
display(pd.Series({name: kurtosis(image).item() for name, image in images.items()}, name="excess kurtosis").to_frame())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=[16, 8])
for a, pair in zip(ax, (("Log luminance", "Random phase"), ("Whitened", "Whitened, random phase"))):
    standard = [host(images[name] / images[name].std(correction=0)) for name in pair]
    density = [a.hist(s.ravel(), bins="auto", density=True, histtype="step", log=True, label=name)[0] for name, s in zip(pair, standard)]
    reach = max(np.abs(s).max() for s in standard)
    grid = np.linspace(-reach, reach)
    a.plot(grid, np.exp(-grid ** 2 / 2) / np.sqrt(2 * np.pi), "k--", lw=1, label="Gaussian")
    a.plot(grid, np.exp(-np.sqrt(2) * np.abs(grid)) / np.sqrt(2), "k:", lw=1, label="Laplace")
    a.set_ylim(bottom=min(d[d > 0].min() for d in density))
    a.set_title(pair[0])
    a.set_xlabel("Standard deviations"), a.set_ylabel("Density")
    a.legend()
    a.set_box_aspect(1), a.grid(alpha=0.25)

---
# PCA
The main directions of colour variation: mostly overall brightness, then colour differences, and where each shows in the image.

- Runs on linear beauty, not the encoded image, so axes reflect radiance rather than display gamma
- PC1 close to (1, 1, 1)/√3 means variation is mostly brightness; PC2 and PC3 then carry chroma
- Projection is each pixel's coordinate along that axis; distance is its distance from that axis (norm over the other two)

In [ ]:
def principal_components(img: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """PCA of (..., c) samples: mean, variances and unit axes (columns) largest first, and each sample's (..., c) projection."""
    pixels = img.reshape(-1, img.shape[-1])
    mean = pixels.mean(0)
    variance, axes = torch.linalg.eigh(torch.cov(pixels.T))
    variance, axes = variance.flip(0), axes.flip(1)
    # eigh signs are arbitrary: make each axis's largest loading positive so projections are reproducible
    axes = axes * axes.gather(0, axes.abs().argmax(0, keepdim=True)).sign()
    return mean, variance, axes, ((pixels - mean) @ axes).reshape(img.shape)

In [ ]:
_, variance, axes, projection = principal_components(rgb)
display(pd.DataFrame(host(axes.T), index=("PC1", "PC2", "PC3"), columns=("R", "G", "B")).assign(variance=host(variance), explained=host(variance / variance.sum())))

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=[18, 7])
for i in range(3):
    ax[0, i].imshow(host(projection[..., i]))
    ax[0, i].set_title(f"PC{i + 1} projection")
    distance = torch.cat((projection[..., :i], projection[..., i + 1:]), -1).norm(dim=-1)
    ax[1, i].imshow(host(distance), cmap="gray")
    ax[1, i].set_title(f"PC{i + 1} distance")
for a in ax.flat: a.set_xticks([]), a.set_yticks([])
plt.tight_layout()

---
# ICA
Learns small detectors whose outputs are as independent as possible: on natural images they become oriented edge detectors, like V1 simple cells.

- Patches are WHITENING's receptive field window, a gain-peak wavelength each side of centre, so their size is measured rather than chosen
- Hex rows repeat every two, so patches start on even rows only and all share one geometry; square patches start on every row
- Whitened by patch PCA, not WHITENING's filter: FastICA needs exactly sphered input (Hyvärinen & Oja 2000), and K leaves `R / (R + N)`, not 1
- A component is kept where its signal `λ - n` beats its noise `n`, WHITENING's R > N test per component; `n` is the halves' half-difference
- The rest would be fitted to render noise: this is van Hateren & van der Schaaf's (1998) PCA dimension reduction, with a measured cut
- Symmetric FastICA with `g = tanh`, the contrast `log cosh` (Hyvärinen 1999): robust for the super-Gaussian sources SPARSITY found
- ICA filters come out localised, oriented and band-pass, Gabor-like (Bell & Sejnowski 1997; van Hateren & van der Schaaf 1998)
- PCA `Λ^-1/2 Uᵀ`, ZCA `U Λ^-1/2 Uᵀ` and ICA `W Λ^-1/2 Uᵀ` whiten equally; ICA's rotation `W` alone seeks sparse responses, so its kurtosis is highest
- The window spans a few pitches at the default 64x32, so filters are pixel-scale edges and corners; larger renders widen it to show gratings

In [ ]:
def independent_components(z: torch.Tensor, generator: torch.Generator) -> torch.Tensor:
    """Symmetric FastICA with g = tanh (Hyvärinen 1999) of sphered (n, d) data: the orthogonal (d, d) unmixing, one component per row."""
    def decorrelate(w: torch.Tensor) -> torch.Tensor:
        # (W Wᵀ)^-1/2 W, the nearest orthogonal matrix: every component is updated in parallel, none favoured (Hyvärinen & Oja 2000 §6.2)
        s, u = torch.linalg.eigh(w @ w.T)
        return (u / s.sqrt()) @ u.T @ w

    w = torch.linalg.qr(torch.randn(z.shape[1], z.shape[1], generator=generator, device=z.device)).Q
    # Fixed point up to sign: each row's cosine with its predecessor within sqrt(eps) of 1, MINPACK's default tolerance
    tolerance = math.sqrt(torch.finfo(z.dtype).eps)
    while True:
        y = torch.tanh(z @ w.T)
        update = decorrelate(y.T @ z / len(z) - (1 - y ** 2).mean(0)[:, None] * w)
        converged = 1 - (update * w).sum(1).abs().min().item() < tolerance
        w = update
        if converged: return w

In [ ]:
row_step = 2 if row_shift else 1
patches = lambda x: x.unfold(0, field.shape[0], row_step).unfold(1, field.shape[1], 1).reshape(-1, field.numel())
_, patch_variance, patch_axes, patch_projection = principal_components(patches(log_luminance))
component_noise = (patches(halves[0] - halves[1]) / 2 @ patch_axes).var(0, correction=1)
keep = patch_variance > 2 * component_noise
sphering = patch_axes[:, keep] / patch_variance[keep].sqrt()
z = patch_projection[:, keep] / patch_variance[keep].sqrt()

unmixing = independent_components(z, torch.Generator(device=DEVICE).manual_seed(0))
unmixing = unmixing[kurtosis(z @ unmixing.T, 0).argsort(descending=True)]
rotations = {"PCA": torch.eye(z.shape[1], device=DEVICE), "ZCA": patch_axes[:, keep].T, "ICA": unmixing.T}
filters = {name: (sphering @ r).T for name, r in rotations.items()}
kurtoses = {name: kurtosis(z @ r, 0) for name, r in rotations.items()}

print(f"{len(z)} patches of {field.shape[0]}x{field.shape[1]}: {keep.sum().item()} of {keep.numel()} components above the noise")
display(pd.DataFrame({name: pd.Series(host(k)) for name, k in kurtoses.items()}).agg(["median", "mean"]).T.rename_axis("excess kurtosis"))

In [ ]:
def mosaic(f):
    tile = (field.shape[0] + 1, field.shape[1] + 1)
    columns = math.ceil(math.sqrt(len(f) * tile[0] * row_pitch / tile[1]))
    rows = -(-len(f) // columns)
    tiles = torch.nn.functional.pad((f / f.abs().amax(1, keepdim=True)).reshape(-1, *field.shape), (0, 1, 0, 1, 0, rows * columns - len(f)))
    return lattice_preview(host(tiles.reshape(rows, columns, *tile).permute(0, 2, 1, 3).reshape(rows * tile[0], columns * tile[1], 1)), rendercam)

previews = {name: mosaic(f) for name, f in filters.items()}
fig, ax = plt.subplots(1, 4, figsize=[30, 8], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews.values()), 1])
for a, (name, p) in zip(ax, previews.items()):
    a.imshow(p, cmap="gray", vmin=-1, vmax=1)
    a.set_title(f"{name} filters")
    a.set_xticks([]), a.set_yticks([])

for name, k in kurtoses.items():
    ax[3].plot(np.linspace(0, 1, k.numel()), host(k.sort(descending=True).values), lw=1, label=name)
ax[3].set_title("Response kurtosis")
ax[3].set_xlabel("Rank (fraction of components)"), ax[3].set_ylabel("Excess kurtosis")
ax[3].legend()
ax[3].set_box_aspect(1), ax[3].grid(alpha=0.25)

---
# SPARSE CODING
Rebuilds each patch from as few parts as possible, out of a dictionary with twice as many parts as dimensions: it learns ICA's kind of edges.

- Generative model `z = Φa + ε` (Olshausen & Field 1996, 1997): Gaussian noise of variance σ² and a Laplace prior of scale b on each coefficient
- σ² is measured, the mean noise fraction of ICA's kept components; b matches the prior's energy `K · 2b²` to the signal's `d(1 - σ²)`
- MAP coefficients solve the lasso `½‖z - Φa‖² + (σ²/b)‖a‖₁` (Tibshirani 1996) by FISTA (Beck & Teboulle 2009): most are exactly zero
- Each atom is refitted exactly with the others fixed, then projected onto the unit ball (Mairal et al. 2010), so there is no learning rate
- Both loops stop once the log posterior gains under 1 nat, a posterior ratio below e: "barely worth mentioning" (Jeffreys 1961; Kass & Raftery 1995)
- K = 2d is the smallest integer overcompleteness, which keeps several patches per atom at the default 64x32
- ICA is the noiseless, complete limit: one filter per dimension, inverted exactly; sparse coding infers a few atoms per patch, the rest silent
- Both live in whitened space: each ICA basis vector (a row of W) is matched by |cos| to its nearest atom, and each atom to its nearest ICA vector
- Atoms with no close ICA vector are what overcompleteness adds; a residual below σ² shows the extra atoms also absorb some render noise

In [ ]:
def sparse_coding(z: torch.Tensor, atoms: int, noise: float, generator: torch.Generator) -> Tuple[torch.Tensor, torch.Tensor]:
    """Olshausen & Field (1997) MAP sparse coding of whitened (n, d) data, noise per dimension: (d, atoms) dictionary, (n, atoms) codes."""
    d = z.shape[1]
    # Laplace scale b by moment matching: unit-variance z carries signal energy d(1 - noise) = atoms · 2b² over unit-norm atoms
    scale = math.sqrt(d * (1 - noise) / (2 * atoms))
    # MAP coefficients solve the lasso ½‖z - Φa‖² + (noise / b)‖a‖₁; dividing by noise gives the negative log posterior in nats
    penalty = noise / scale
    nats = lambda codes, dictionary: ((0.5 * ((z - codes @ dictionary.T) ** 2).sum() + penalty * codes.abs().sum()) / noise).item()
    dictionary = torch.randn(d, atoms, generator=generator, device=z.device)
    dictionary /= dictionary.norm(dim=0)
    codes = z.new_zeros(len(z), atoms)
    previous = math.inf
    while True:
        # FISTA (Beck & Teboulle 2009) warm-started from the last codes, step 1 / ‖Φ‖₂², the gradient's Lipschitz constant
        gram, target = dictionary.T @ dictionary, z @ dictionary
        step = 1 / torch.linalg.eigvalsh(gram)[-1]
        search, momentum, inner = codes, 1.0, nats(codes, dictionary)
        while True:
            update = search - step * (search @ gram - target)
            update = update.sign() * (update.abs() - step * penalty).clamp(min=0)
            accelerated = (1 + math.sqrt(1 + 4 * momentum ** 2)) / 2
            search = update + (momentum - 1) / accelerated * (update - codes)
            codes, momentum = update, accelerated
            value = nats(codes, dictionary)
            # Under 1 nat is a posterior ratio below e, within "not worth more than a bare mention" (Jeffreys 1961; Kass & Raftery 1995)
            if inner - value < 1: break
            inner = value
        # Checked after inference, so the codes returned are MAP for the dictionary returned
        if previous - value < 1: return dictionary, codes
        previous = value
        energy, correlation = codes.T @ codes, z.T @ codes
        # Each used atom's exact minimiser, rest fixed, on the unit ball (Mairal et al. 2010, Alg. 2); unused atoms keep their place
        for j in energy.diag().nonzero().flatten().tolist():
            atom = dictionary[:, j] + (correlation[:, j] - dictionary @ energy[:, j]) / energy[j, j]
            dictionary[:, j] = atom / atom.norm().clamp(min=1)

In [ ]:
noise_fraction = (component_noise[keep] / patch_variance[keep]).mean().item()
dictionary, codes = sparse_coding(z, 2 * z.shape[1], noise_fraction, torch.Generator(device=DEVICE).manual_seed(0))
order = codes.abs().mean(0).argsort(descending=True)
dictionary, codes = dictionary[:, order], codes[:, order]
overlap = (unmixing @ (dictionary / dictionary.norm(dim=0))).abs()
residual = ((z - codes @ dictionary.T) ** 2).mean().item()

codes_by_model = {"ICA": z @ unmixing.T, "Sparse coding": codes}
print(f"{dictionary.shape[1]} atoms for {z.shape[1]} dimensions; residual {residual:.3f} per dimension vs noise {noise_fraction:.3f}")
display(pd.DataFrame({
    "median excess kurtosis": [kurtosis(c, 0).quantile(0.5).item() for c in codes_by_model.values()],
    "exact zeros": [(c == 0).float().mean().item() for c in codes_by_model.values()],
}, index=list(codes_by_model)))

In [ ]:
unwhitening = patch_axes[:, keep] * patch_variance[keep].sqrt()
previews = {"ICA basis": mosaic((unwhitening @ unmixing.T).T), "Sparse coding atoms": mosaic((unwhitening @ dictionary).T)}
fig, ax = plt.subplots(1, 3, figsize=[30, 8], width_ratios=[*(p.shape[1] / p.shape[0] for p in previews.values()), 1])
for a, (name, p) in zip(ax, previews.items()):
    a.imshow(p, cmap="gray", vmin=-1, vmax=1)
    a.set_title(name)
    a.set_xticks([]), a.set_yticks([])

ax[2].hist(host(overlap.amax(1)), bins="auto", range=(0, 1), histtype="step", label="ICA vector to nearest atom")
ax[2].hist(host(overlap.amax(0)), bins="auto", range=(0, 1), histtype="step", label="Atom to nearest ICA vector")
ax[2].set_title("Best match")
ax[2].set_xlabel("|cos|"), ax[2].set_ylabel("Vectors")
ax[2].legend()
ax[2].set_box_aspect(1), ax[2].grid(alpha=0.25)